# QAT M-LSD tiny (bản 2) trên Colab

Chọn **Runtime > Change runtime type > T4 GPU** trước khi chạy.

Notebook chạy `8-qat/mlsd/qat_mlsd_v2.py` trong repo Quantization: so PTQ và QAT trên cùng cấu hình (tách head, giữ 3 layer nhạy ở float, ai-edge-quantizer static INT8).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Lấy code và dữ liệu

In [ ]:
%cd /content
!rm -rf Quantization
!git clone -q https://github.com/TruongDuke/Quantization.git
# M-LSD bản PyTorch (có sẵn weight .pth)
!git clone -q https://github.com/lhwcv/mlsd_pytorch.git Quantization/5-mlsd/mlsd-pytorch
# Imagenette (ảnh train / calibration / đánh giá)
!wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz -O /content/imagenette.tgz
!tar xzf /content/imagenette.tgz -C Quantization/4-quantization/
!ls Quantization/5-mlsd/mlsd-pytorch/models Quantization/4-quantization/imagenette2-160

## 2. Cài thư viện (đúng phiên bản như máy local)

Lệnh này cài lại torch 2.13. Nếu chạy xong Colab báo cần **Restart session** thì bấm restart, rồi chạy tiếp từ bước 3 (không cần chạy lại bước 1, 2).

In [ ]:
!pip install -q litert-torch==0.9.4 ai-edge-quantizer==0.9.0 ai-edge-litert torchvision==0.28.0 opencv-python-headless
import torch, torchvision
print(torch.__version__, torchvision.__version__, torch.cuda.is_available())

## 3. Chạy QAT

1000 bước, batch 8, có thêm bản QAT quantize tất cả layer để so sánh. Trên T4 ước lượng 15-30 phút.

In [ ]:
%cd /content/Quantization/8-qat/mlsd
!python qat_mlsd_v2.py --steps 1000 --bs 8 --also_full_int8 2>&1 | grep -E "^(Device|FP32|PTQ|QAT|\[keep3\]|\[full\]|Xong)|Traceback|Error"

## 4. Xem kết quả và tải về

In [ ]:
import json
r = json.load(open('/content/Quantization/8-qat/mlsd/results/qat_mlsd_v2.json'))
print(f"{'Ban':26s} {'center':>7s} {'Dmed':>6s} {'giu':>6s} {'MB':>5s} {'ms':>6s}")
for x in r['rows']:
    print(f"{x['name']:26s} {x['center_recall']:7.1f} {x['disp_err_median_px']:6.2f} {x['lines_kept']:6.1f} {x['size_mb']:5.2f} {x['ms']:6.1f}")

In [ ]:
from google.colab import files
files.download('/content/Quantization/8-qat/mlsd/results/qat_mlsd_v2.json')
files.download('/content/Quantization/8-qat/mlsd/models/v2_qat_keep3.tflite')